[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/heitorramos/icd/blob/main/exemplos/21-regressao-multipla/notebook-colab.ipynb)


In [ ]:
# Preparação automática para execução no Google Colab.
# Fora do Colab, esta célula não altera o diretório de trabalho.
try:
    import google.colab  # type: ignore
except ImportError:
    pass
else:
    import os
    import subprocess
    from pathlib import Path

    repository = Path("/content/icd")
    if not repository.exists():
        subprocess.run([
            "git", "clone", "--depth", "1",
            "https://github.com/heitorramos/icd.git", str(repository)
        ], check=True)
    os.chdir(repository / "exemplos/21-regressao-multipla")
    print("Material preparado em:", Path.cwd())


# Regressão linear múltipla e introdução a GLMs

Material de apoio — Aula 21

## Objetivos

Este guia formula regressão múltipla em notação matricial, implementa o
ajuste e interpreta seus coeficientes. Ao final, introduz os três
componentes de um modelo linear generalizado para preparar a regressão
logística.

## Como estudar este capítulo

A regressão múltipla amplia a reta simples para situações em que várias
características ajudam a explicar a resposta. A novidade mais importante
não é apenas adicionar colunas à matriz: a interpretação passa a ser
**condicional**. O coeficiente de idade descreve a variação associada à
idade quando os demais atributos do modelo são mantidos fixos.

O capítulo começa pela construção da matriz de projeto, porque cada
coluna corresponde a uma escolha de representação: variável numérica,
indicador de categoria ou interação. Depois ajustamos o modelo,
interpretamos os coeficientes e investigamos dois desafios: interações,
nas quais o efeito de uma variável depende de outra, e colinearidade,
que dificulta separar contribuições individuais.

Use a notação matricial como um mapa compacto. Cada linha de $X$
representa uma observação, cada coluna um atributo e $\beta$ contém um
coeficiente por coluna. O produto $X\beta$ produz todas as médias
previstas de uma vez. A intuição continua sendo a mesma da regressão
simples.

## Base de dados de apoio

Usaremos [Medical Insurance
Cost](https://www.kaggle.com/datasets/mosapabdelghany/medical-insurance-cost-dataset),
do Kaggle. Cada linha representa uma pessoa segurada.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
rng = np.random.default_rng(20260827)
df = pd.read_csv(Path("../16-correlacao/data/insurance.csv"))
df.head()

In [ ]:
df.info()

In [ ]:
df.describe(include="all").T

> **Interpretação**
>
> A base combina variáveis quantitativas e categóricas. `charges` é
> assimétrica, tabagismo separa patamares e as regiões possuem quatro
> níveis. Essas propriedades determinam a codificação e orientam o
> diagnóstico.

## Modelo estatístico

Para $i=1,\ldots,n$,

$$
Y_i=\beta_0+\sum_{j=1}^p\beta_jx_{ij}+\varepsilon_i,
\qquad E[\varepsilon_i\mid X_i]=0.
$$

$i$ indexa observações, $j$ indexa preditores, $x_{ij}$ é o valor do
preditor $j$ para a observação $i$ e $\beta_j$ é seu coeficiente
populacional.

$\beta_j$ compara a média prevista após aumentar $X_j$ em uma unidade,
mantendo fixos os demais preditores do modelo.

> **Interpretação**
>
> “Manter fixo” define uma comparação condicional, não uma intervenção.
> Variáveis omitidas, erro de mensuração e forma funcional inadequada
> ainda podem impedir uma leitura causal.

## Codificação da matriz de projeto

Usaremos `northeast` como referência regional e `female`, `no smoker`
como referências dos indicadores binários.

In [ ]:
Xdf = pd.DataFrame({
    "age": df["age"],
    "bmi": df["bmi"],
    "children": df["children"],
    "smoker_yes": (df["smoker"] == "yes").astype(int),
    "sex_male": (df["sex"] == "male").astype(int),
})
Xdf = pd.concat([
    Xdf,
    pd.get_dummies(df["region"], prefix="region", drop_first=True, dtype=int)
], axis=1)
Xdf.head()

> **Interpretação**
>
> Remover uma categoria evita dependência linear perfeita entre o
> intercepto e os indicadores. Alterar a referência muda a
> parametrização, mas não muda os valores ajustados.

## Forma matricial

Com uma coluna de 1 para o intercepto,

$$E[y\mid X]=X\beta,$$

em que $X$ é $n\times(p+1)$, $\beta$ é $(p+1)\times1$ e $y$ é
$n\times1$.

In [ ]:
X = np.column_stack([np.ones(len(df)), Xdf.to_numpy(float)])
y = df["charges"].to_numpy(float)
names = ["intercepto"] + list(Xdf.columns)
pd.Series({"linhas_X": X.shape[0], "colunas_X": X.shape[1],
           "tamanho_y": len(y), "posto_X": np.linalg.matrix_rank(X)})

O posto igual ao número de colunas confirma que os coeficientes estão
identificados nesta codificação.

## Perda e gradiente

$$J(b)=\|y-Xb\|_2^2=(y-Xb)^T(y-Xb).$$

Expandindo:

$$J(b)=y^Ty-2b^TX^Ty+b^TX^TXb.$$

Logo,

$$\nabla_bJ(b)=-2X^T(y-Xb).$$

O componente $j$ é $-2\sum_i x_{ij}e_i$: resíduos ponderados pela coluna
$j$.

## Ajuste numérico

Usamos `lstsq`, que resolve o problema sem formar explicitamente
$(X^TX)^{-1}$.

In [ ]:
beta = np.linalg.lstsq(X, y, rcond=None)[0]
coef = pd.Series(beta, index=names, name="coeficiente")
coef.round(2)

> **Interpretação**
>
> Mantendo os demais preditores fixos, um ano adicional de idade
> corresponde a cerca de US\$ 256,86, uma unidade de IMC a US\$ 339,19 e
> ser fumante a aproximadamente US\$ 23.848,54 na média prevista. Essas
> são associações condicionais do modelo.

## Valores ajustados e qualidade

In [ ]:
prediction = X @ beta
residual = y-prediction
sse = np.sum(residual**2)
sst = np.sum((y-y.mean())**2)
pd.Series({
    "R²": 1-sse/sst,
    "RMSE": np.sqrt(np.mean(residual**2)),
    "média_resíduos": residual.mean(),
}).round(4)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].scatter(y, prediction, s=10, alpha=.35)
axes[0].plot([y.min(), y.max()], [y.min(), y.max()], "--", color="darkorange")
axes[0].set(xlabel="observado", ylabel="ajustado")
axes[1].scatter(prediction, residual, s=10, alpha=.35)
axes[1].axhline(0, color="black", linestyle="--")
axes[1].set(xlabel="ajustado", ylabel="resíduo")
plt.tight_layout(); plt.show()

> **Interpretação**
>
> O modelo múltiplo explica muito mais variação que idade isolada,
> sobretudo pela inclusão de tabagismo. Ainda há assimetria, valores
> extremos e variância residual desigual; mais $R^2$ não significa
> modelo plenamente adequado.

## Interações

Para permitir inclinações de idade distintas por tabagismo:

$$
E[Y]=\beta_0+\beta_1age+\beta_2smoker
+\beta_3(age\times smoker)+\cdots.
$$

In [ ]:
X_inter = Xdf.copy()
X_inter["age_x_smoker"] = X_inter["age"]*X_inter["smoker_yes"]
Xi = np.column_stack([np.ones(len(df)), X_inter.to_numpy(float)])
bi = np.linalg.lstsq(Xi, y, rcond=None)[0]
pd.Series(bi, index=["intercepto"]+list(X_inter.columns)).loc[
    ["age", "smoker_yes", "age_x_smoker"]
].round(2)

> **Interpretação**
>
> Com interação, o coeficiente de idade é a inclinação entre não
> fumantes; `age_x_smoker` é quanto essa inclinação muda entre fumantes.
> Termos principais devem permanecer para preservar a hierarquia do
> modelo.

## Colinearidade

Colinearidade ocorre quando colunas de $X$ são altamente relacionadas.
Ela pode inflar a incerteza dos coeficientes sem destruir
necessariamente a qualidade preditiva conjunta.

In [ ]:
numeric_corr = Xdf[["age", "bmi", "children"]].corr()
numeric_corr.round(3)

Uma forma diagnóstica é observar a sensibilidade dos coeficientes a
pequenas mudanças na amostra.

## Bootstrap do vetor de coeficientes

In [ ]:
B = 1000
boot = np.empty((B, X.shape[1]))
for b in range(B):
    idx = rng.integers(0, len(df), len(df))
    boot[b] = np.linalg.lstsq(X[idx], y[idx], rcond=None)[0]

summary = pd.DataFrame({
    "estimativa": beta,
    "q2.5": np.quantile(boot, .025, axis=0),
    "q97.5": np.quantile(boot, .975, axis=0),
}, index=names)
summary.round(2)

> **Interpretação**
>
> Cada intervalo mantém fixa a especificação do modelo e quantifica a
> variação sob reamostragem das pessoas observadas. Ele não incorpora
> incerteza sobre quais variáveis deveriam entrar nem corrige vieses do
> desenho observacional.

## Validação e complexidade

Adicionar preditores nunca aumenta a SSE de treinamento. Por isso, $R^2$
não diminui. Para avaliar generalização, precisamos separar dados de
treinamento e validação ou usar validação cruzada.

Mais colunas podem melhorar a aproximação, mas também aumentar
variância, instabilidade e risco de capturar ruído.

## Ponte para modelos lineares generalizados

Um GLM possui:

1.  componente aleatório: distribuição de $Y_i\mid X_i$ na família
    exponencial;
2.  componente sistemático: $\eta_i=x_i^T\beta$;
3.  função de ligação: $g(\mu_i)=\eta_i$, com $\mu_i=E[Y_i\mid X_i]$.

## O modelo normal é um GLM

Na regressão linear normal:

$$Y_i\mid X_i\sim N(\mu_i,\sigma^2),$$

$$g(\mu_i)=\mu_i=x_i^T\beta.$$

O link identidade deixa a média na mesma escala do preditor linear.

## Antecipação da regressão logística

Para resposta binária,

$$Y_i\mid X_i\sim\operatorname{Bernoulli}(\pi_i),$$

$$\log\left(\frac{\pi_i}{1-\pi_i}\right)=x_i^T\beta.$$

Logo,

$$\pi_i=\frac{1}{1+e^{-x_i^T\beta}},$$

o que garante probabilidades entre 0 e 1.

> **Interpretação**
>
> A regressão logística preserva a matriz $X$ e o preditor linear
> $X\beta$. O que muda é a distribuição da resposta, a função de ligação
> e, consequentemente, a verossimilhança e a perda.

## Como interpretar o modelo sem se perder nos coeficientes

O modelo múltiplo é uma extensão da reta: continuamos descrevendo uma
média, mas agora com várias informações simultâneas. Um roteiro seguro
é:

1.  **Definir a resposta e a unidade observacional.** Neste exemplo,
    cada linha é uma pessoa e a resposta são suas despesas.
2.  **Escolher atributos disponíveis para a pergunta.** Idade, IMC e
    tabagismo entram por razões diferentes e possuem unidades
    diferentes.
3.  **Codificar categorias.** Uma categoria vira referência; os
    coeficientes das demais expressam diferenças em relação a ela.
4.  **Ajustar o modelo.** O algoritmo encontra a combinação de
    coeficientes que reduz os resíduos.
5.  **Interpretar um coeficiente por vez.** A leitura sempre inclui
    “mantendo os demais atributos do modelo constantes”.
6.  **Verificar se essa comparação é plausível.** Combinações raras de
    atributos podem levar a interpretações apoiadas por poucos dados.
7.  **Analisar resíduos e validação.** Um bom coeficiente isolado não
    substitui a avaliação do modelo completo.

### Variável categórica

Se `smoker=yes` é comparado com `smoker=no`, seu coeficiente representa
a diferença média prevista entre fumantes e não fumantes com a mesma
idade e o mesmo IMC, segundo o modelo. Isso é uma comparação
condicional, não automaticamente um efeito causal.

### Interação

Uma interação responde a perguntas como: “a relação entre IMC e despesas
muda entre fumantes e não fumantes?”. Ao incluí-la, passamos a ter uma
inclinação para o grupo de referência e uma correção dessa inclinação
para o outro grupo.

## Colinearidade em linguagem prática

Quando dois atributos trazem quase a mesma informação, o modelo pode
prever bem, mas ter dificuldade para decidir quanto do efeito atribuir a
cada coeficiente. Sinais podem mudar e erros-padrão podem aumentar.
Antes de retirar variáveis mecanicamente, volte à pergunta: precisamos
de previsão ou de interpretação separada dos coeficientes?

> **Leitura recomendada**
>
> Leia primeiro a previsão produzida para perfis concretos e só depois
> examine os coeficientes. Isso reduz o risco de interpretar um termo
> fora da escala, da categoria de referência ou das interações do
> modelo.

## Questões de revisão

1.  Explique as dimensões de $X$, $\beta$, $y$ e $\nabla J$.
2.  Derive $\nabla J=-2X^T(y-X\beta)$.
3.  Interprete um coeficiente indicador e uma interação.
4.  Por que mudar a categoria de referência não muda previsões?
5.  Como colinearidade afeta interpretação e previsão?
6.  Quais são os três componentes de um GLM?

## Bibliografia

- James et al., *An Introduction to Statistical Learning*, capítulos 3 e
  4.
- Faraway, *Linear Models with R*.
- Gelman, Hill e Vehtari, *Regression and Other Stories*.
- Fox, *Applied Regression Analysis and Generalized Linear Models*.
- McCullagh e Nelder, *Generalized Linear Models*.